In [1]:
import json
import numpy as np
from pymargo.core import Engine
import pyyokan_common as yokan
from pyyokan_client import Client
from pyyokan_server import Provider

In [2]:
# Params
provider_id = 123
protocol = 'na+sm'
server_addr = 'na+sm://1953185-0'

In [3]:
# init engine
engine = Engine(protocol)
mid = engine.get_internal_mid()
addr = engine.lookup(server_addr)
hg_addr = addr.get_internal_hg_addr()
provider = Provider(mid=mid, provider_id=provider_id, config='{"database":{"type":"map"}}')
client = Client(mid=mid)
db = client.make_database_handle(address=hg_addr, provider_id=provider_id)

In [4]:
def serialize(data):
    if isinstance(data, np.ndarray):
        ser_data = np.array2string(data, separator=',')
        return ser_data, type(data[0]), len(data)
    else:
        ser_data = str(data)
        return ser_data, type(data), 1    

In [5]:
def deserialize(str_data, num_elems, data_type):
    if num_elems == 1:
        if data_type == 'float':
            return np.float32(str_data)
        elif data_type == 'double':
            return np.float64(str_data)
        elif data_type == 'int':
            return np.int(str_data)
    else:
        serialized_data = str_data[1:-1]
        orig = np.fromstring(serialized_data, dtype=data_type, sep=',')
        return orig

In [6]:
def list_keys(db, num_keys):
    max_length = 1024
    prefix = ''
    out_keys = []
    for i in range(0, num_keys):
      out_keys.append( bytearray(max_length+len(prefix)+1) )
    
    from_key = ''
    ksizes = db.list_keys(keys=out_keys, from_key=from_key, filter=prefix)
    
    keys = []
    for i in range(len(ksizes)):
        key_size = ksizes[i]
        
        k = out_keys[i]
        key = (k[:key_size]).decode('ascii')
        keys.append(key)
        
    return keys

In [7]:
db.put("_4/field/energy/value", (serialize(np.array([0, 1, 2, 3])))[0]  )
db.put("_4/field/energy/rank", "0"  )
db.put("_4/field/energy/num_elems", (serialize(4))[0])
db.put("_4/field/energy/type", "float")

In [8]:
db.put("_4/field/energy/value", (serialize(np.array([0.1, 1.1, 2.1, 3.1, 4.1])))[0]  )
db.put("_4/field/energy/rank", "1" )
db.put("_4/field/energy/num_elems", (serialize(5))[0])
db.put("_4/field/energy/type", "float")

In [9]:
db.put("_4/field/pressure/value", (serialize(np.array([0, 1, 2, 3])))[0]  )
db.put("_4/field/pressure/rank", "0"  )
db.put("_4/field/pressure/num_elems", (serialize(4))[0])
db.put("_4/field/pressure/type", "float")

In [10]:
db.put("_4/field/pressure/value", (serialize(np.array([1.1, 2.1, 3.1, 4.1, 5.1])))[0] )
db.put("_4/field/energy/rank", "1"  )
db.put("_4/field/pressure/num_elems", (serialize(5))[0])
db.put("_4/field/pressure/type", "float")

In [11]:
db.put("_4/state/time/value", (serialize(1.04))[0] )
db.put("_4/state/time/num_elems", '1')
db.put("_4/state/time/type", "float")

In [12]:
num_keys = db.count()
num_keys

11

11

In [13]:
keys = list_keys(db, db.count())
print(keys)

['_4/field/energy/num_elems', '_4/field/energy/rank', '_4/field/energy/type', '_4/field/energy/value', '_4/field/pressure/num_elems', '_4/field/pressure/rank', '_4/field/pressure/type', '_4/field/pressure/value', '_4/state/time/num_elems', '_4/state/time/type', '_4/state/time/value']
['_4/field/energy/num_elems', '_4/field/energy/rank', '_4/field/energy/type', '_4/field/energy/value', '_4/field/pressure/num_elems', '_4/field/pressure/rank', '_4/field/pressure/type', '_4/field/pressure/value', '_4/state/time/num_elems', '_4/state/time/type', '_4/state/time/value']


In [14]:
ts = '_4'
result = list(filter(lambda x: x.startswith(ts), keys))
print(result)

['_4/field/energy/num_elems', '_4/field/energy/rank', '_4/field/energy/type', '_4/field/energy/value', '_4/field/pressure/num_elems', '_4/field/pressure/rank', '_4/field/pressure/type', '_4/field/pressure/value', '_4/state/time/num_elems', '_4/state/time/type', '_4/state/time/value']
['_4/field/energy/num_elems', '_4/field/energy/rank', '_4/field/energy/type', '_4/field/energy/value', '_4/field/pressure/num_elems', '_4/field/pressure/rank', '_4/field/pressure/type', '_4/field/pressure/value', '_4/state/time/num_elems', '_4/state/time/type', '_4/state/time/value']


In [15]:
name = "energy"
result = list(filter(lambda x: x.count(name), result))
print(result)

['_4/field/energy/num_elems', '_4/field/energy/rank', '_4/field/energy/type', '_4/field/energy/value']
['_4/field/energy/num_elems', '_4/field/energy/rank', '_4/field/energy/type', '_4/field/energy/value']


In [16]:
data_type_key = ( list(filter(lambda x: x.endswith('num_elems'),      result)) )[0]
data_type_key

'_4/field/energy/num_elems'

'_4/field/energy/num_elems'

In [ ]:
data_type_key = ( list(filter(lambda x: x.endswith('type'),      result)) )[0]
num_elems     = ( list(filter(lambda x: x.endswith('num_elems'), result)) )[0]
query_key     = ( list(filter(lambda x: x.endswith('value'), result)) )[0]

In [ ]:
query_key

In [ ]:
ts = str(4)

_keys = []
for k in keys:
    if k.startswith(ts)

In [ ]:
def get_field_name(key):
    parts = keys[0].split('/')
    name = parts[len(parts)-2]
    return name

In [ ]:
name = get_field_name(keys[0])
print(name)

In [17]:
def get_data(db, key):
    ''' Get data from the server for that key '''

    # length of the value associated with the key
    l = db.length(key)

    out_val = bytearray(l)          # create buffer
    db.get(key=key, value=out_val)  # get the data
    v = out_val.decode("ascii")     # convert to ascii
    return v

In [30]:
def get_value(db, field, ts):
    
    # Get the list of keys
    num_keys = db.count()
    keys = list_keys(db, num_keys)
    print(keys)
    
    # Filter by name and timestep
    result_1 = list( filter(lambda x: x.startswith( '_' + str(ts) ), keys) )
    result_2 = list( filter(lambda x: x.count(field)  , result_1) )
    
    data_type_key = ( list(filter(lambda x: x.endswith('type'),      result_2)) )[0]
    num_elems_key = ( list(filter(lambda x: x.endswith('num_elems'), result_2)) )[0]
    query_key     = ( list(filter(lambda x: x.endswith('value'), result_2)) )[0]
    rank_key      = ( list(filter(lambda x: x.endswith('rank'), result_2)) )[0]
    print("num_ranks",rank_key)
    
    str_data  = get_data(db, query_key)
    num_elems = get_data(db, num_elems_key)
    data_type = get_data(db, data_type_key)
    ranks = get_data(db, rank_key)
    print("ranks",ranks)
    
    data = deserialize(str_data, int(num_elems), data_type)
    return data

In [31]:
data = get_value(db,'energy',4)
data

['_4/field/energy/num_elems', '_4/field/energy/rank', '_4/field/energy/type', '_4/field/energy/value', '_4/field/pressure/num_elems', '_4/field/pressure/rank', '_4/field/pressure/type', '_4/field/pressure/value', '_4/state/time/num_elems', '_4/state/time/type', '_4/state/time/value']
num_ranks _4/field/energy/rank
ranks 1


array([0.1, 1.1, 2.1, 3.1, 4.1])

['_4/field/energy/num_elems', '_4/field/energy/rank', '_4/field/energy/type', '_4/field/energy/value', '_4/field/pressure/num_elems', '_4/field/pressure/rank', '_4/field/pressure/type', '_4/field/pressure/value', '_4/state/time/num_elems', '_4/state/time/type', '_4/state/time/value']
num_ranks _4/field/energy/rank
ranks 1


array([0.1, 1.1, 2.1, 3.1, 4.1])